# L8 — VWAP Volume Baselines: Ejercicios

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | 1–5 | Completar en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o alumnos adelantados |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'monospace', 'axes.facecolor': '#18181b',
                     'figure.facecolor': '#09090b', 'axes.edgecolor': '#27272a',
                     'grid.color': '#27272a', 'text.color': '#e4e4e7'})

CYAN   = '#22d3ee'
GREEN  = '#4ade80'
RED    = '#f87171'
AMBER  = '#f59e0b'
PURPLE = '#a78bfa'
MUTED  = '#a1a1aa'

---
## Ejercicio 0 — Reflexión (sin código)

Antes de abrir ningún CSV, lee este código:

In [ ]:
# Un trader quiere comprar 10 BTC ahora mismo
ask_prices = [100_000, 100_010, 100_020, 100_030, 100_040,
              100_050, 100_060, 100_070, 100_080, 100_090]
ask_sizes  = [0.5, 0.8, 1.2, 0.9, 1.5, 2.0, 1.8, 1.3, 2.5, 1.9]
# Total disponible: sum(ask_sizes) = 14.4 BTC en 10 niveles

print("Reflexiona sobre estas preguntas:")
print("1. Si compras los 10 BTC de golpe, ¿a qué precio promedio compras?")
print("   (Sugerencia: pondera precio × tamaño nivel a nivel hasta completar 10 BTC)")
print()
print("2. Si el volumen de BTC es 2.28× más alto a las 00:00 que a las 12:00,")
print("   ¿cuándo tiene más sentido ejecutar para minimizar impacto?")
print()
print("3. ¿Qué información necesitarías para construir un schedule que 'siga el mercado'?")

In [ ]:
# Respuesta a la pregunta 1 — slippage de una orden de 10 BTC
total_to_buy = 10.0
remaining = total_to_buy
total_cost = 0.0

for price, size in zip(ask_prices, ask_sizes):
    fill = min(remaining, size)
    total_cost += fill * price
    remaining -= fill
    if remaining <= 0:
        break

vwap_exec = total_cost / total_to_buy
slippage_per_btc = vwap_exec - ask_prices[0]

print(f'VWAP ejecutado (10 BTC de golpe): ${vwap_exec:,.2f}')
print(f'Slippage: ${slippage_per_btc:.2f}/BTC')
print(f'Coste total del slippage: ${slippage_per_btc * total_to_buy:.2f}')

---
## Ejercicio 1 — Núcleo: Cargar y explorar los datos

In [ ]:
# Carga '../data/btc_volume_intraday.csv' con parse_dates=['datetime']
# Asigna el DataFrame a la variable `df`
# Calcula n_rows (int) y n_days (int)

df = None  # TODO
n_rows = None  # TODO
n_days = None  # TODO

print(df)

In [ ]:
# Validador E1
assert 'df' in dir() and df is not None, "Asigna el DataFrame a 'df'"
assert isinstance(df, pd.DataFrame), "df debe ser un pandas DataFrame"
assert n_rows == 6048, f"n_rows debe ser 6048, tienes {n_rows}. ¿Cargaste el CSV correcto?"
assert n_days == 21, f"n_days debe ser 21, tienes {n_days}"
assert 'volume_normalized' in df.columns, "El CSV debe tener la columna 'volume_normalized'"
assert 'interval_idx' in df.columns, "El CSV debe tener la columna 'interval_idx'"
print("✓ E1 correcto — 6048 filas, 21 días")

In [ ]:
# Solución E1
df = pd.read_csv('../data/btc_volume_intraday.csv', parse_dates=['datetime'])
n_rows = len(df)
n_days = df['date'].nunique()
print(f'Cargado: {n_rows:,} filas, {n_days} días ({df["date"].min()} → {df["date"].max()})')
print(df.dtypes)

---
## Ejercicio 2 — Núcleo: Volúmenes diarios y efecto DOW

In [ ]:
# Calcula daily_volumes: suma del volumen (columna 'volume') por fecha
# Tipo esperado: pd.Series con índice = date
#
# Luego calcula:
#   mean_daily_vol  — media de daily_volumes (float)
#   monday_mean_vol — media de daily_volumes solo para lunes (weekday==0)

daily_volumes = None   # TODO
mean_daily_vol = None  # TODO
monday_mean_vol = None # TODO

print(daily_volumes)

In [ ]:
# Validador E2
assert daily_volumes is not None, "Calcula daily_volumes"
assert abs(mean_daily_vol - 25152.2) < 200, \
    f"mean_daily_vol debe ser ~25152, tienes {mean_daily_vol:.1f}"
assert monday_mean_vol < mean_daily_vol, \
    "Los lunes tienen menos volumen que la media global — revisa el filtro weekday==0"
assert monday_mean_vol < 23000, \
    f"monday_mean_vol debe ser ~21290, tienes {monday_mean_vol:.0f}"
print(f"✓ E2 correcto — media diaria: {mean_daily_vol:,.0f} BTC, lunes: {monday_mean_vol:,.0f} BTC")

In [ ]:
# Solución E2
daily_volumes = df.groupby('date')['volume'].sum()
mean_daily_vol = float(daily_volumes.mean())

monday_dates = df[df['weekday'] == 0]['date'].unique()
monday_mean_vol = float(daily_volumes[daily_volumes.index.isin(monday_dates)].mean())

print(f'Media diaria global: {mean_daily_vol:,.0f} BTC')
print(f'Media diaria lunes:  {monday_mean_vol:,.0f} BTC')
print(f'Ratio lunes/global:  {monday_mean_vol/mean_daily_vol:.2f}')

---
## Ejercicio 3 — Núcleo: Construir el perfil medio

In [ ]:
# Calcula mean_profile: media de 'volume_normalized' agrupado por interval_idx
# Tipo esperado: pd.Series con índice = interval_idx (0 a 287)
#
# Luego calcula:
#   profile_sum      — suma del perfil (float, debe ser ~1.0)
#   open_close_ratio — mean_profile[0] / mean_profile[144] (float)

mean_profile = None    # TODO
profile_sum = None     # TODO
open_close_ratio = None  # TODO

print(mean_profile)

In [ ]:
# Validador E3
assert mean_profile is not None, "Calcula mean_profile"
assert isinstance(mean_profile, pd.Series), "mean_profile debe ser una pd.Series"
assert len(mean_profile) == 288, f"El perfil debe tener 288 valores, tienes {len(mean_profile)}"
assert abs(profile_sum - 1.0) < 1e-4, \
    f"El perfil debe sumar 1.0 (fracción del volumen diario). Suma actual: {profile_sum:.6f}"
assert abs(open_close_ratio - 2.28) < 0.05, \
    f"La ratio open/midday debe ser ~2.28, tienes {open_close_ratio:.4f}"
print(f"✓ E3 correcto — perfil suma {profile_sum:.6f}, open/midday ratio: {open_close_ratio:.4f}×")

In [ ]:
# Solución E3
mean_profile = df.groupby('interval_idx')['volume_normalized'].mean()
profile_sum = float(mean_profile.sum())
open_close_ratio = float(mean_profile.iloc[0] / mean_profile.iloc[144])

# Plot
fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(range(288), mean_profile.values, alpha=0.3, color=CYAN)
ax.plot(mean_profile.values, color=CYAN, linewidth=1.5)
xticks = [0, 72, 144, 216, 287]
ax.set_xticks(xticks)
ax.set_xticklabels(['00:00', '06:00', '12:00', '18:00', '23:55'])
ax.set_title(f'Perfil medio — forma U | open/midday ratio: {open_close_ratio:.2f}×', color=CYAN)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Ejercicio 4 — Núcleo: `build_vwap_schedule`

In [ ]:
# Implementa build_vwap_schedule(total_qty, volume_profile)
# La función debe:
#   1. Normalizar volume_profile para que sume 1.0
#   2. Multiplicar por total_qty
#   3. Retornar una pd.Series con los BTC a ejecutar en cada intervalo
#
# Luego llama: schedule_10btc = build_vwap_schedule(10.0, mean_profile)

def build_vwap_schedule(total_qty: float, volume_profile: pd.Series) -> pd.Series:
    pass  # TODO

schedule_10btc = None  # TODO

print(schedule_10btc)

In [ ]:
# Validador E4
assert callable(build_vwap_schedule), "build_vwap_schedule debe ser una función"
assert schedule_10btc is not None, "Llama a build_vwap_schedule y asigna el resultado"
assert isinstance(schedule_10btc, pd.Series), "El resultado debe ser una pd.Series"
assert len(schedule_10btc) == 288, f"El schedule debe tener 288 elementos, tienes {len(schedule_10btc)}"
assert abs(schedule_10btc.sum() - 10.0) < 1e-6, \
    f"El schedule debe sumar exactamente 10 BTC, suma actual: {schedule_10btc.sum():.6f}"
assert abs(schedule_10btc.iloc[0] - 0.04812157) < 1e-4, \
    f"schedule[0] debe ser ~0.04812, tienes {schedule_10btc.iloc[0]:.6f}. ¿Normalizaste bien el perfil?"
print(f"✓ E4 correcto — schedule suma {schedule_10btc.sum():.4f} BTC, intervalo 0: {schedule_10btc.iloc[0]:.5f} BTC")

In [ ]:
# Solución E4
def build_vwap_schedule(total_qty: float, volume_profile: pd.Series) -> pd.Series:
    """
    Distribuye total_qty sobre los intervalos de volume_profile,
    proporcional al volumen de cada intervalo.
    """
    normalized = volume_profile / volume_profile.sum()
    return total_qty * normalized

schedule_10btc = build_vwap_schedule(10.0, mean_profile)

print(f'Intervalo más activo: {schedule_10btc.argmax():3d} → {schedule_10btc.max():.4f} BTC')
print(f'Intervalo más quieto: {schedule_10btc.argmin():3d} → {schedule_10btc.min():.4f} BTC')
print(f'Total: {schedule_10btc.sum():.6f} BTC')

---
## Ejercicio 5 — Núcleo: Los 4 baselines

In [ ]:
# Construye los 4 perfiles baseline y guárdalos en un dict llamado `profiles`:
#
#   'mean_all':      media de volume_normalized por interval_idx (todos los días)
#   'median_all':    mediana de volume_normalized por interval_idx (todos los días)
#   'mean_monday':   media filtrada a weekday==0
#   'median_monday': mediana filtrada a weekday==0
#
# Cada perfil es una pd.Series con índice = interval_idx y len = 288

profiles = {}  # TODO — añade las 4 claves

print(profiles)

In [ ]:
# Validador E5
required = ['mean_all', 'median_all', 'mean_monday', 'median_monday']
for key in required:
    assert key in profiles, f"Falta '{key}' en profiles"
    assert isinstance(profiles[key], pd.Series), f"profiles['{key}'] debe ser pd.Series"
    assert len(profiles[key]) == 288, f"profiles['{key}'] debe tener 288 elementos"

assert abs(profiles['mean_all'].sum() - 1.0) < 1e-4, \
    "mean_all debe sumar 1.0"
assert abs(profiles['mean_monday'].iloc[0] - 0.00406789) < 1e-4, \
    f"mean_monday[0] debe ser ~0.00407 (lunes tiene open más bajo que la media global). Tienes {profiles['mean_monday'].iloc[0]:.6f}"
assert profiles['mean_monday'].iloc[0] < profiles['mean_all'].iloc[0], \
    "El open del lunes debe ser menor que el open medio global — los lunes abren con menos volumen"

print(f"✓ E5 correcto — 4 baselines construidos")
print(f"  mean_all[0]:    {profiles['mean_all'].iloc[0]:.6f}")
print(f"  mean_monday[0]: {profiles['mean_monday'].iloc[0]:.6f} (lunes: {profiles['mean_monday'].iloc[0]/profiles['mean_all'].iloc[0]*100:.1f}% del global)")

In [ ]:
# Solución E5
profiles = {
    'mean_all':      df.groupby('interval_idx')['volume_normalized'].mean(),
    'median_all':    df.groupby('interval_idx')['volume_normalized'].median(),
    'mean_monday':   df[df['weekday'] == 0].groupby('interval_idx')['volume_normalized'].mean(),
    'median_monday': df[df['weekday'] == 0].groupby('interval_idx')['volume_normalized'].median(),
}

print(f'{"Baseline":<20} {"sum":>10} {"[0]":>12}')
print('-' * 44)
for name, p in profiles.items():
    print(f'{name:<20} {p.sum():>10.6f} {p.iloc[0]:>12.8f}')

---
## Ejercicio 6 — Si vamos bien: `rmse_profile`

In [ ]:
# Implementa rmse_profile(pred_profile, eval_df):
#   - Para cada día en eval_df, extrae el perfil real (volume_normalized por interval_idx)
#   - Compara con pred_profile usando MSE
#   - Retorna RMSE global (float)
#
# Luego calcula: rmse_mean_all = rmse_profile(profiles['mean_all'], df)

def rmse_profile(pred_profile: pd.Series, eval_df: pd.DataFrame) -> float:
    pass  # TODO

rmse_mean_all = None  # TODO

print(rmse_mean_all)

In [ ]:
# Validador E6
assert callable(rmse_profile), "rmse_profile debe ser una función"
assert rmse_mean_all is not None, "Calcula rmse_mean_all"
assert abs(rmse_mean_all - 0.00052448) < 1e-6, \
    f"rmse_mean_all debe ser 0.00052448. Tienes {rmse_mean_all:.8f}.\n" \
    "Pista: ¿Estás iterando sobre cada día en eval_df y comparando contra pred_profile?"
print(f"✓ E6 correcto — RMSE mean_all: {rmse_mean_all:.8f}")

In [ ]:
# Solución E6
def rmse_profile(pred_profile: pd.Series, eval_df: pd.DataFrame) -> float:
    """
    RMSE entre el perfil predicho y los perfiles reales de cada día en eval_df.
    Opera sobre volume_normalized.
    """
    errors = []
    pred = pred_profile.values
    for date, day_df in eval_df.groupby('date'):
        actual = day_df.sort_values('interval_idx')['volume_normalized'].values
        errors.extend((actual - pred) ** 2)
    return float(np.sqrt(np.mean(errors)))

rmse_mean_all = rmse_profile(profiles['mean_all'], df)
print(f'RMSE mean_all (21 días): {rmse_mean_all:.8f}')

---
## Ejercicio 7 — Si vamos bien: Evaluar los 4 baselines

In [ ]:
# Calcula rmse_results: dict {nombre_baseline: rmse_valor} para los 4 baselines
# evaluados contra TODOS los días del dataset.
#
# Luego identifica:
#   best_baseline        — nombre del baseline con menor RMSE global (str)
#   best_baseline_monday — nombre del baseline con menor RMSE solo en lunes (str)
#                          (evalúa filtrando df a weekday==0)

rmse_results = {}         # TODO
best_baseline = None      # TODO
best_baseline_monday = None  # TODO

print(rmse_results)

In [ ]:
# Validador E7
assert len(rmse_results) == 4, "rmse_results debe tener 4 entradas"
assert best_baseline == 'mean_all', \
    f"El mejor baseline global es 'mean_all'. Tienes '{best_baseline}'."
assert best_baseline_monday == 'mean_monday', \
    f"El mejor baseline para lunes es 'mean_monday'. Tienes '{best_baseline_monday}'."

rmse_monday_check = rmse_profile(profiles['mean_monday'], df[df['weekday']==0])
assert abs(rmse_monday_check - 0.00042372) < 1e-5, \
    f"RMSE mean_monday en lunes debe ser ~0.00042372. Tienes {rmse_monday_check:.8f}"

print(f"✓ E7 correcto")
print(f"  Best global: {best_baseline} ({rmse_results[best_baseline]:.8f})")
print(f"  Best lunes:  {best_baseline_monday} ({rmse_monday_check:.8f})")
print(f"  Mejora en lunes: {(rmse_results['mean_all'] - rmse_monday_check)/rmse_results['mean_all']*100:.1f}%")

In [ ]:
# Solución E7
rmse_results = {name: rmse_profile(prof, df) for name, prof in profiles.items()}
best_baseline = min(rmse_results, key=rmse_results.get)

monday_eval = df[df['weekday'] == 0]
rmse_monday = {name: rmse_profile(prof, monday_eval) for name, prof in profiles.items()}
best_baseline_monday = min(rmse_monday, key=rmse_monday.get)

print('RMSE global:')
for name, rmse in sorted(rmse_results.items(), key=lambda x: x[1]):
    marker = ' ← mejor' if name == best_baseline else ''
    print(f'  {name:<20} {rmse:.8f}{marker}')

print('\nRMSE solo lunes:')
for name, rmse in sorted(rmse_monday.items(), key=lambda x: x[1]):
    marker = ' ← mejor (lunes)' if name == best_baseline_monday else ''
    print(f'  {name:<20} {rmse:.8f}{marker}')

---
## Ejercicio 8 — Bonus: Visualizar los 4 schedules

In [ ]:
# Crea una figura con 4 subplots (2×2).
# En cada subplot: el schedule de 10 BTC para uno de los 4 baselines.
# Añade una línea horizontal punteada en 10.0/288 (flat/uniforme).
# Añade etiquetas de hora en el eje X: 00:00, 06:00, 12:00, 18:00.
# Cada subplot tiene el nombre del baseline como título.
#
# Usa los colores: CYAN, GREEN, RED, AMBER para cada subplot.

colors_list = [CYAN, GREEN, RED, AMBER]
xtick_positions = [0, 72, 144, 216, 287]
xtick_labels = ['00:00', '06:00', '12:00', '18:00', '23:55']
flat_level = 10.0 / 288

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)

for ax, (name, prof), col in zip(axes.flat, profiles.items(), colors_list):
    sched = None  # TODO: build_vwap_schedule(10.0, prof)
    # TODO: fill_between, plot, axhline, set_title, set_xticks, grid
    pass

fig.suptitle('VWAP Schedules — 10 BTC en 4 baselines', color=CYAN, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Validador E8 (light — solo comprueba que creaste una figura)
assert plt.get_fignums(), "Debes crear una figura matplotlib con plt.subplots(2, 2)"
print("✓ E8 — figura creada. Revisa visualmente que:")
print("  1. Los 4 subplots tienen curvas distintas (no todas iguales)")
print("  2. La línea punteada flat está debajo de los picos")
print("  3. El subplot mean_monday tiene el pico del open más bajo que mean_all")

In [ ]:
# Solución E8
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)
flat_level = 10.0 / 288

for ax, (name, prof), col in zip(axes.flat, profiles.items(), colors_list):
    sched = build_vwap_schedule(10.0, prof)
    ax.fill_between(range(288), sched.values, alpha=0.25, color=col)
    ax.plot(sched.values, color=col, linewidth=1.5)
    ax.axhline(flat_level, color=MUTED, linewidth=0.8, linestyle='--', label='flat')
    ax.set_title(name, color=col, fontweight='bold', fontsize=10)
    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_labels, fontsize=8)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

fig.suptitle('VWAP Schedules — 10 BTC en 4 baselines (vs flat)', 
             color=CYAN, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Ejercicio 9 — Bonus: `select_best_profile`

In [ ]:
# Implementa select_best_profile(dow, all_profiles):
#   - dow: int (0=lunes, 1=martes, ..., 6=domingo)
#   - all_profiles: dict con los perfiles disponibles
#   - Si existe 'mean_{día}' en all_profiles, retorna ese perfil
#   - Si no existe, retorna all_profiles['mean_all']
#
# Luego:  best_schedule = build_vwap_schedule(10.0, select_best_profile(0, profiles))

def select_best_profile(dow: int, all_profiles: dict) -> pd.Series:
    pass  # TODO

best_schedule = None  # TODO

print(best_schedule)

In [ ]:
# Validador E9
assert callable(select_best_profile), "select_best_profile debe ser una función"
assert best_schedule is not None, "Calcula best_schedule para dow=0"
assert abs(best_schedule.sum() - 10.0) < 1e-6, "El schedule debe sumar 10.0 BTC"
assert abs(best_schedule.iloc[0] - 0.04067894) < 2e-4, \
    f"Para dow=0 (lunes), best_schedule[0] debe ser ~0.04068. Tienes {best_schedule.iloc[0]:.6f}\n" \
    "Pista: ¿Estás usando el perfil mean_monday (no mean_all) para los lunes?"
print(f"✓ E9 correcto — lunes usa mean_monday, schedule[0]: {best_schedule.iloc[0]:.5f} BTC")

# Comprobación fallback
fallback = select_best_profile(3, profiles)  # jueves — no existe mean_thursday
assert (fallback == profiles['mean_all']).all(), "Para días sin perfil específico, debe usar mean_all"
print("  Fallback correcto — jueves usa mean_all")

In [ ]:
# Solución E9
def select_best_profile(dow: int, all_profiles: dict) -> pd.Series:
    """
    Selecciona el mejor baseline según el día de la semana.
    Usa el perfil específico del día si existe, mean_all como fallback.
    """
    DOW_NAMES = {0: 'monday', 1: 'tuesday', 2: 'wednesday',
                 3: 'thursday', 4: 'friday', 5: 'saturday', 6: 'sunday'}
    key = f'mean_{DOW_NAMES[dow]}'
    return all_profiles.get(key, all_profiles['mean_all'])

best_schedule = build_vwap_schedule(10.0, select_best_profile(0, profiles))

print(f'Lunes: open={best_schedule.iloc[0]:.4f} BTC vs global open={build_vwap_schedule(10.0, profiles["mean_all"]).iloc[0]:.4f} BTC')
print(f'El lunes abre con {best_schedule.iloc[0]/build_vwap_schedule(10.0, profiles["mean_all"]).iloc[0]*100:.1f}% del volumen global')

---
## Ejercicio 10 — Bonus: Análisis de sensibilidad

In [ ]:
# Para cada total_qty en [1, 5, 10, 20, 50, 100], calcula el schedule usando mean_profile.
# Guarda en sensitivity_df (DataFrame) con columnas:
#   'total_qty', 'max_interval_qty', 'flat_interval_qty', 'concentration_ratio'
#
# concentration_ratio = max_interval_qty / flat_interval_qty
#   (cuánto más compras en el intervalo pico vs una distribución uniforme)
#
# Hipótesis a verificar: ¿cambia concentration_ratio con total_qty?

qty_range = [1, 5, 10, 20, 50, 100]

sensitivity_df = None  # TODO

print(sensitivity_df)

In [ ]:
# Validador E10
assert isinstance(sensitivity_df, pd.DataFrame), "sensitivity_df debe ser un DataFrame"
assert len(sensitivity_df) == 6, f"Debe tener 6 filas, tienes {len(sensitivity_df)}"
assert 'concentration_ratio' in sensitivity_df.columns, "Falta la columna 'concentration_ratio'"

row_10 = sensitivity_df[sensitivity_df['total_qty'] == 10]['concentration_ratio'].iloc[0]
assert abs(row_10 - 2.28) < 0.05, \
    f"Para total_qty=10, concentration_ratio debe ser ~2.28. Tienes {row_10:.4f}"

# La ratio no debe cambiar con total_qty
ratios = sensitivity_df['concentration_ratio'].values
assert np.std(ratios) < 0.001, \
    "concentration_ratio no debe variar con total_qty — es una propiedad del perfil, no del tamaño"

print(f"✓ E10 correcto — concentration_ratio constante: {ratios.mean():.4f}×")
print("\nInsight: la concentración es independiente del tamaño total.")
print("El perfil determina la FORMA del schedule, no su escala.")

In [ ]:
# Solución E10
rows = []
for qty in qty_range:
    sched = build_vwap_schedule(qty, mean_profile)
    max_qty = float(sched.max())
    flat_qty = qty / 288
    rows.append({
        'total_qty': qty,
        'max_interval_qty': round(max_qty, 6),
        'flat_interval_qty': round(flat_qty, 6),
        'concentration_ratio': round(max_qty / flat_qty, 4)
    })

sensitivity_df = pd.DataFrame(rows)
print(sensitivity_df.to_string(index=False))

print(f"\nSTD de concentration_ratio: {sensitivity_df['concentration_ratio'].std():.8f}")
print("→ La ratio es constante. El perfil es escala-libre.")